# Benchmark for MS MARCO 8.8M on Kaggle

**Instructions:**
1. Ensure your Kaggle notebook is using a **CPU-only environment** (no GPU) to maximize available RAM (30GB).
2. Upload your `hnsw-code.zip` file as a dataset (name it `hnsw-code`).
3. Attach your `msmarco-8.8M.hdf5` dataset.
4. Attach your query dataset containing `msmarco_qemb_train.npz` and `msmarco_qemb_validation.npz`.
5. Update the dataset paths in the configuration cell below.
6. Run the notebook from top to bottom. The index will be saved to `/kaggle/working/` and can be downloaded from the Output tab.

In [ ]:
!pip install pybind11

# IMPORTANT: Adjust the path below if your uploaded dataset has a different name than 'hnsw-code'
!mkdir -p /kaggle/working/chao_hybrid_ada_ef
!cp -r /kaggle/input/hnsw-code/* /kaggle/working/chao_hybrid_ada_ef/
!cd /kaggle/working/chao_hybrid_ada_ef && python setup.py build_ext --inplace

import sys
sys.path.append('/kaggle/working/chao_hybrid_ada_ef')

In [ ]:
import os, sys, time
import h5py
import numpy as np
from scipy.spatial.distance import cdist
from scipy.stats import norm, entropy as sp_entropy, spearmanr
from sklearn.cluster import MiniBatchKMeans

import chao_hybrid_ada_ef_cpp

# Compute ground truth via exhaustive search
def compute_ground_truth(corpus, queries, k=10):
    print(f'Computing ground truth for {len(queries)} queries...')
    gt = np.zeros((len(queries), k), dtype=int)
    batch_size = 100
    for i in range(0, len(queries), batch_size):
        q_batch = queries[i:i+batch_size]
        dists = cdist(q_batch, corpus, metric='sqeuclidean')
        gt[i:i+batch_size] = np.argsort(dists, axis=1)[:, :k]
    return gt

np.random.seed(42)

In [ ]:
# --- Configuration ---
K_SEARCH       = 10
TARGET_RECALL  = 0.95
# Extended sweep for larger corpus
EF_SWEEP       = [10, 30, 50, 100, 200, 400, 600, 800, 1000, 1500, 2000]
N_CALIB        = 2000     
S_PROBES       = 500      # Increased for 8.8M
K_CLUSTERS     = 500      # Increased for 8.8M

# --- File Paths ---
# UPDATE THESE to point to your attached Kaggle datasets
CORPUS_PATH = '/kaggle/input/your-dataset-name/msmarco-8.8M.hdf5'
TRAIN_Q_PATH = '/kaggle/input/your-queries-dataset/msmarco_qemb_train.npz'
TEST_Q_PATH = '/kaggle/input/your-queries-dataset/msmarco_qemb_validation.npz'

INDEX_PATH = '/kaggle/working/custom_8.8M.index'

In [ ]:
def build_ef_table(scores_int, required_efs):
    table = {}
    for s in np.unique(scores_int):
        table[int(s)] = int(np.percentile(required_efs[scores_int == s], 90))
    return table

def lookup_ef(score, table, min_ef=10, max_ef=2000):
    if not table: return max_ef
    if score in table: return int(np.clip(table[score], min_ef, max_ef))
    known = sorted(table.keys())
    if score <= known[0]: return int(np.clip(table[known[0]], min_ef, max_ef))
    if score >= known[-1]: return int(np.clip(table[known[-1]], min_ef, max_ef))
    lo = max(k for k in known if k <= score)
    hi = min(k for k in known if k >= score)
    if lo == hi: return int(np.clip(table[lo], min_ef, max_ef))
    frac = (score - lo) / (hi - lo)
    return int(np.clip(table[lo] + frac * (table[hi] - table[lo]), min_ef, max_ef))

Z_QUANTILES = [norm.ppf(0.2), norm.ppf(0.4), norm.ppf(0.6), norm.ppf(0.8)]

def ada_ef_score(queries, samp_vecs, mean_v, cov_v):
    mu_ip = queries @ mean_v
    mu_l2 = 2 - 2 * mu_ip
    sig_ip_sq = np.sum((queries @ cov_v) * queries, axis=1)
    sig_l2 = 2 * np.sqrt(np.clip(sig_ip_sq, 0, None))
    z = np.array(Z_QUANTILES)[None, :]
    bins = mu_l2[:, None] + z * sig_l2[:, None]
    probe_dists = cdist(queries, samp_vecs, metric='sqeuclidean')
    scores = (probe_dists[:, :, None] > bins[:, None, :]).sum(axis=(1, 2))
    return scores.astype(np.float64)

In [ ]:
print('═' * 80)
print('  Loading Dataset')
print('═' * 80)

with h5py.File(CORPUS_PATH, 'r') as f:
    # Find the dataset key automatically
    dataset_key = list(f.keys())[0]
    # Loading as float32 to save memory
    corpus = f[dataset_key][:].astype(np.float32)

train_q_full = np.load(TRAIN_Q_PATH)['emb'].astype(np.float32)
test_q = np.load(TEST_Q_PATH)['emb'].astype(np.float32)

dim = corpus.shape[1]
print(f'  Corpus: {corpus.shape} | Train Q: {train_q_full.shape} | Test Q: {test_q.shape} | dim={dim}')

calib_q = train_q_full[np.random.choice(len(train_q_full), N_CALIB, replace=False)]

print('\nComputing ground truth (this might take a few minutes)...')
t0 = time.time()
calib_gt = compute_ground_truth(corpus, calib_q, k=K_SEARCH)
test_gt  = compute_ground_truth(corpus, test_q,  k=K_SEARCH)
print(f'  Done in {time.time() - t0:.1f}s')

In [ ]:
print('\nBuilding HNSW index (this may take 1-2 hours for 8.8M vectors)...')
idx = chao_hybrid_ada_ef_cpp.Index(space='l2', dim=dim)

if os.path.exists(INDEX_PATH):
    print(f'Loading existing index from {INDEX_PATH}...')
    try:
        idx.load_index(INDEX_PATH, max_elements=corpus.shape[0])
    except Exception as e:
        print(f'Failed to load: {e}')
        print('Rebuilding instead...')
        idx.init_index(max_elements=corpus.shape[0], ef_construction=200, M=16)
        idx.add_items(corpus)
        idx.save_index(INDEX_PATH)
else:
    idx.init_index(max_elements=corpus.shape[0], ef_construction=200, M=16)
    idx.add_items(corpus)
    idx.save_index(INDEX_PATH)
    print(f'Index built and saved to {INDEX_PATH}')

In [ ]:
print(f'\n{"═" * 80}')
print(f'  Shared Calibration: min ef for {N_CALIB} queries (target={TARGET_RECALL})')
print(f'{"═" * 80}')

t0 = time.time()
calib_min_ef = np.zeros(N_CALIB, dtype=np.float32)
for i in range(N_CALIB):
    for ef in EF_SWEEP:
        labs, _ = idx.search_knn_adaptive(calib_q[i], K_SEARCH, idx.entry_point, idx.max_level, ef)
        rec = len(set(labs) & set(calib_gt[i])) / K_SEARCH
        if rec >= TARGET_RECALL:
            calib_min_ef[i] = ef
            break
    else:
        calib_min_ef[i] = EF_SWEEP[-1]
    if (i + 1) % 500 == 0:
        print(f'  ... {i + 1}/{N_CALIB}')

print(f'  Done in {time.time() - t0:.1f}s')
print(f'  Required ef: mean={calib_min_ef.mean():.0f}, p90={np.percentile(calib_min_ef, 90):.0f}')

In [ ]:
print(f'\n{"═" * 80}\n  ADA-EF: Offline Phase\n{"═" * 80}')
t_ada_total = time.time()

corpus_mean = np.mean(corpus, axis=0)
sub = corpus[np.random.choice(len(corpus), min(100_000, len(corpus)), replace=False)]
corpus_cov = np.cov(sub, rowvar=False).astype(np.float32)
samp_vectors = corpus[np.random.choice(len(corpus), S_PROBES, replace=False)]

ada_calib_scores = ada_ef_score(calib_q, samp_vectors, corpus_mean, corpus_cov)
ada_scores_int = np.round(ada_calib_scores).astype(int)
ada_table = build_ef_table(ada_scores_int, calib_min_ef)

ada_corr = spearmanr(ada_calib_scores, calib_min_ef).correlation
print(f'  EF table size: {len(ada_table)} | Score-ef Spearman ρ = {ada_corr:.3f}')

In [ ]:
print(f'\n{"═" * 80}\n  CLUSTER-AWARE: Offline Phase\n{"═" * 80}')
t_clust_total = time.time()

km = MiniBatchKMeans(n_clusters=K_CLUSTERS, random_state=42, n_init=3, batch_size=10000)
km.fit(corpus)
centroids = km.cluster_centers_.astype(np.float32)
labels = km.labels_

Z_QUANTILES_PCT = [20, 40, 60, 80]
cluster_bins = np.zeros((K_CLUSTERS, 4), dtype=np.float32)
for k in range(K_CLUSTERS):
    pts = corpus[labels == k]
    if len(pts) > 0:
        dists = cdist(pts, centroids[k:k+1], metric='sqeuclidean').flatten()
        cluster_bins[k] = np.percentile(dists, Z_QUANTILES_PCT)
    else:
        cluster_bins[k] = np.array([0.5, 1.0, 1.5, 2.0])

calib_cdists = cdist(calib_q, centroids, metric='sqeuclidean')
calib_nearest = np.argmin(calib_cdists, axis=1)

clust_calib_scores = np.zeros(N_CALIB, dtype=np.float32)
for i in range(N_CALIB):
    k_id = calib_nearest[i]
    bins = cluster_bins[k_id].tolist()
    clust_calib_scores[i] = idx.get_dynamic_probe_score(calib_q[i], bins, 20)

clust_table = build_ef_table(np.round(clust_calib_scores).astype(int), calib_min_ef)
max_score = max(clust_table.keys()) if clust_table else 0
ef_table_list = [lookup_ef(s, clust_table, max_ef=2000) for s in range(max_score + 1)] if clust_table else [10]

clust_corr = spearmanr(clust_calib_scores, calib_min_ef).correlation
print(f'  EF table size: {len(clust_table)} | Score-ef Spearman ρ = {clust_corr:.3f}')

In [ ]:
n_test = len(test_q)

def eval_vanilla(name, ef):
    idx.reset_dist_count()
    recs = []
    t0 = time.time()
    for i in range(n_test):
        labs, _ = idx.search_knn_adaptive(test_q[i], K_SEARCH, idx.entry_point, idx.max_level, ef)
        recs.append(len(set(labs) & set(test_gt[i])) / K_SEARCH)
    r = np.array(recs)
    return dict(name=name, mean_r=np.mean(r), hnsw_dc=idx.get_dist_count()/n_test, time=time.time()-t0)

def eval_ada_ef():
    idx.reset_dist_count()
    recs, efs = [], []
    t0 = time.time()
    test_ada_scores = ada_ef_score(test_q, samp_vectors, corpus_mean, corpus_cov)
    test_ada_int = np.round(test_ada_scores).astype(int)
    for i in range(n_test):
        ef = lookup_ef(test_ada_int[i], ada_table, max_ef=2000)
        efs.append(ef)
        labs, _ = idx.search_knn_adaptive(test_q[i], K_SEARCH, idx.entry_point, idx.max_level, ef)
        recs.append(len(set(labs) & set(test_gt[i])) / K_SEARCH)
    r = np.array(recs)
    return dict(name='Ada-ef', mean_r=np.mean(r), hnsw_dc=idx.get_dist_count()/n_test, time=time.time()-t0, avg_ef=np.mean(efs))

def eval_cluster_aware():
    idx.reset_dist_count()
    recs = []
    t0 = time.time()
    test_cdists = cdist(test_q, centroids, metric='sqeuclidean')
    test_nearest = np.argmin(test_cdists, axis=1)
    for i in range(n_test):
        k_id = test_nearest[i]
        labs, _ = idx.search_knn_dynamic(test_q[i], K_SEARCH, cluster_bins[k_id].tolist(), ef_table_list, 10, 2000, 20)
        recs.append(len(set(labs) & set(test_gt[i])) / K_SEARCH)
    r = np.array(recs)
    return dict(name='Cluster-Aware', mean_r=np.mean(r), hnsw_dc=idx.get_dist_count()/n_test, time=time.time()-t0)

results = []
for ef in [100, 400, 800, 1500]:
    print(f'Vanilla(ef={ef})...')
    results.append(eval_vanilla(f'Vanilla(ef={ef})', ef))

print('Ada-ef...')
results.append(eval_ada_ef())

print('Cluster-Aware...')
results.append(eval_cluster_aware())

print('\n' + '='*80)
print(f'{ "Method":<20} {"Recall":<10} {"DC":<15} {"Time (s)":<10}')
print('='*80)
for r in results:
    print(f'{r["name"]:<20} {r["mean_r"]:<10.4f} {r["hnsw_dc"]:<15.0f} {r["time"]:<10.2f}')